# Chimera on real fraud - baselines + the closed loop on public data

This notebook grounds Chimera on a **real public benchmark** (the ULB credit-card fraud dataset, 284,807 genuine transactions, via OpenML). It runs standard baselines against Chimera's two-channel ensemble, then applies the **red-team/blue-team loop to real fraud** - perturbing real fraud to evade the model, then retraining to recover. This shows the methodology is not tied to our synthetic simulator.

Repo: https://github.com/dhruv-decoder/chimera  ·  Enable Internet in the notebook settings.

In [ ]:
!git clone -q https://github.com/dhruv-decoder/chimera.git
%cd chimera
!pip -q install -e "backend[dev]"

### Baselines vs Chimera, and the loop on real data
Runs Logistic Regression, Random Forest, XGBoost (pre-installed on Kaggle) and LightGBM against Chimera's two-channel ensemble, then the evasion/retrain loop on real fraud.

In [ ]:
!cd backend && python scripts/benchmark_baselines.py

In [ ]:
import json, matplotlib.pyplot as plt
r = json.load(open('data/artifacts/benchmark_report.json'))
print('Baselines on real ULB fraud (held-out):')
for k, v in r['baselines'].items():
    print(f"  {k:24s} ROC {v['roc_auc']:.4f}  PR-AUC {v['pr_auc']:.4f}")
cl = r['closed_loop_on_real']
print(f"\nClosed loop on REAL data: baseline {cl['baseline_recall']*100:.0f}%"
      f" -> under evasion {cl['under_evasion']*100:.0f}% -> after retrain {cl['after_retrain']*100:.0f}%")
names = list(r['baselines']); pr = [r['baselines'][n]['pr_auc'] for n in names]
plt.figure(figsize=(7,3.5)); plt.barh(names, pr, color='#2ed6a6')
plt.xlabel('PR-AUC on real ULB fraud'); plt.xlim(0,1); plt.title('Chimera is competitive with standard baselines on real fraud')
plt.gca().invert_yaxis(); plt.tight_layout(); plt.show()

### Takeaway
Chimera's detector is competitive with conventional fraud classifiers on a real, established benchmark - so the contribution is not a marginally-better static classifier, it is the **adaptive red-team/blue-team loop**, which above is shown to break and recover on real fraud, not only on our synthetic simulator.